In [47]:
from sqlalchemy.engine import Engine,URL
from sqlalchemy import (create_engine,text, Table, 
                        Column, Integer, String, Float, Boolean,  Date, DateTime, Time, Text)
from sqlalchemy import MetaData
from dotenv import load_dotenv
from dotenv import load_dotenv
import os
import psycopg2
import pandas as pd
from extract import *
from datetime import datetime, timedelta, date

In [2]:
def create_db_url(username, password, host, port, database):

    source_connection_url = URL.create(
        drivername = 'postgresql+psycopg2',
        username = username,
        password = password,
        host = host,
        port = port,
        database = database
    )

    return source_connection_url

def check_connection(source_connection_url) -> bool:

    try:
        source_engine = create_engine(source_connection_url)
        with source_engine.connect() as connection:
            connection.execute(text("SELECT 1"))
        return True
    except:
        return False

In [3]:
load_dotenv()
username = os.environ.get('DESTINATION_DB_USERNAME')
password = os.environ.get('DESTINATION_DB_PASSWORD')
host = os.environ.get('DESTINATION_SERVER_NAME')
port = os.environ.get('DESTINATION_PORT')
database = os.environ.get('DESTINATION_DATABASE_NAME')

In [4]:
source_url = create_db_url(username,password,host,port,database)
type(source_url)

sqlalchemy.engine.url.URL

In [18]:
if check_connection(source_url):
    print('Connection Successful')
else:
    print('Connection failed! Please check configuration!')

Connection Successful


In [40]:
df = get_data("https://data.cityofnewyork.us/resource/f55k-p6yu.json" ,"persons",'2023-12-04',1000,0)
df

2025-10-06 23:24:45.256 | INFO     | extract:get_data:34 - Saving 928 rows to /mnt/d/dataengineeringcamp/dec_project_01/raw_data/persons/2023-12-04_persons_2025-10-06_23-24-45.csv


,unique_id,collision_id,crash_date,crash_time,person_id,person_type,person_injury,vehicle_id,person_age,ped_role,...,ejection,emotional_status,bodily_injury,position_in_vehicle,safety_equipment,complaint,ped_location,ped_action,contributing_factor_1,contributing_factor_2
0,12826320,4685644,2023-12-04T00:00:00.000,12:28,bfae5a70-f486-4b89-b476-7d15123c29f6,Occupant,Unspecified,20564361,71,Registrant,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12823803,4685130,2023-12-04T00:00:00.000,16:52,a397d979-763e-4152-a41d-ed9a058c76c9,Occupant,Unspecified,20562938,27,Registrant,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,12823241,4684833,2023-12-04T00:00:00.000,4:45,f9aee0d1-ed4d-4deb-9cf3-8e1cde573e44,Occupant,Unspecified,20562610,22,Registrant,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,12823031,4684631,2023-12-04T00:00:00.000,1:30,dd4b7382-fd59-4b28-bab6-45aaa08f3494,Occupant,Unspecified,20562492,NaN,Registrant,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12824300,4685006,2023-12-04T00:00:00.000,15:13,29e2d5fa-d035-4bab-9505-fc9c2cba5a48,Occupant,Unspecified,20563236,NaN,Registrant,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
923,12880463,4684769,2023-12-04T00:00:00.000,9:14,44377ccb-d690-49e6-80a4-84bf7a925a9d,Occupant,Unspecified,20595161,NaN,Registrant,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
924,12880460,4684769,2023-12-04T00:00:00.000,9:14,5efbcdba-8e58-41a9-8da4-9c08aea1f71b,Occupant,Unspecified,NaN,NaN,Notified Person,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
925,13265473,4684466,2023-12-04T00:00:00.000,0:24,f0187864-40d0-4f9f-a055-9cb22cbcdb60,Occupant,Killed,20815354,62,Passenger,...,Ejected,Apparent Death,Head,Right rear passenger or motorcycle sidecar pas...,None,Internal,NaN,NaN,NaN,NaN
926,13265471,4684466,2023-12-04T00:00:00.000,0:24,3dba3267-529f-41f8-9b5a-a21fc5f1275a,Occupant,Injured,20815354,25,Driver,...,Ejected,Conscious,Elbow-Lower-Arm-Hand,Driver,None,Abrasion,NaN,NaN,NaN,NaN


In [43]:
df[['ejection','emotional_status']]

,ejection,emotional_status
0,NaN,NaN
1,NaN,NaN
2,NaN,NaN
3,NaN,NaN
4,NaN,NaN
...,...,...
923,NaN,NaN
924,NaN,NaN
925,Ejected,Apparent Death
926,Ejected,Conscious


In [20]:
def get_type(type_str):

    type_mapping = {
        'Date' : Date,
        'Datetime' : DateTime,
        'String' : String(150),
        'Float' : Float,
        'Integer' : Integer
    }

    return type_mapping.get(type_str, String(150))

In [21]:
def get_table_from_yaml(yaml_file,metadata):

    with open(yaml_file, 'r') as f:
        schema = yaml.safe_load(f)
    
    columns = []

    table_name = list(schema.keys())[0]
    for dict_items in schema[table_name]:
        for column_name, column_type in dict_items.items():
            sql_type = get_type(column_type)
            columns.append(Column(column_name, sql_type))
    
    return Table(table_name, metadata, *columns)

In [29]:
metadata = MetaData()

In [30]:
crashes_table = get_table_from_yaml('./crashes_metadata.yaml',metadata)
crashes_table

Table('mvc_crashes', MetaData(), Column('crash_date', Date(), table=<mvc_crashes>), Column('crash_time', String(length=150), table=<mvc_crashes>), Column('borough', String(length=150), table=<mvc_crashes>), Column('zip_code', String(length=150), table=<mvc_crashes>), Column('latitude', Float(), table=<mvc_crashes>), Column('longitude', Float(), table=<mvc_crashes>), Column('cross_street_name', String(length=150), table=<mvc_crashes>), Column('number_of_persons_injured', Integer(), table=<mvc_crashes>), Column('number_of_persons_killed', Integer(), table=<mvc_crashes>), Column('number_of_pedestrians_injured', Integer(), table=<mvc_crashes>), Column('number_of_pedestrians_killed', Integer(), table=<mvc_crashes>), Column('number_of_cyclist_injured', Integer(), table=<mvc_crashes>), Column('number_of_cyclist_killed', Integer(), table=<mvc_crashes>), Column('number_of_motorist_injured', Integer(), table=<mvc_crashes>), Column('number_of_motorist_killed', Integer(), table=<mvc_crashes>), Col

In [45]:
latest_date = get_date("https://data.cityofnewyork.us/resource/h9gi-nx95.json",'SELECT * ORDER BY crash_date DESC LIMIT 1')
latest_date

datetime.date(2025, 9, 30)

In [48]:
latest_date - timedelta(days=1)

datetime.date(2025, 9, 29)

In [31]:
if check_connection(source_url):
    engine = create_engine(source_url)
    metadata.create_all(engine,checkfirst=True)
    df.to_sql(crashes_table.name, engine, if_exists='append', index=False)
else:
    print('Please check connection')

In [36]:
df.to_sql(crashes_table.name, engine, if_exists='append', index=False)

260

In [16]:
if check_connection(source_url):
    engine = create_engine(source_url)
    sql_query = "SELECT * FROM mvc_crashes"
    df = pd.read_sql(sql_query, engine)
else:
    print('Please check connection')

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271 entries, 0 to 270
Data columns (total 31 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   crash_date                     271 non-null    object
 1   crash_time                     271 non-null    object
 2   borough                        192 non-null    object
 3   zip_code                       192 non-null    object
 4   latitude                       261 non-null    object
 5   longitude                      261 non-null    object
 6   on_street_name                 191 non-null    object
 7   off_street_name                138 non-null    object
 8   number_of_persons_injured      271 non-null    object
 9   number_of_persons_killed       271 non-null    object
 10  number_of_pedestrians_injured  271 non-null    object
 11  number_of_pedestrians_killed   271 non-null    object
 12  number_of_cyclist_injured      271 non-null    object
 13  numbe